In [9]:
import numpy as np
from collections import Counter

In [12]:
class Node:
    def __init__(self,feature, value=None, left=None, right=None, threshold=None):
        self.feature = feature
        self.value = value
        self.left = left
        self.right = right
        self.threshold = threshold

    def is_leaf_node(self):
        return self.value is not None

In [ ]:
class DecisionTree:
    def __init__(self, min_sample_split=5, max_depth=10,n_features=None):
        self.min_sample_split = min_sample_split
        self.max_depth = max_depth
        self.n_features = n_features
        self.root = None     

    def _cross_entropy(self, y):
        histogram = np.bincount(y)
        prob = histogram / len(y)
        return -np.sum([p*np.log(p) for p in prob if p > 0])

    #The below function decides which sample go to
    # the left or right tree. Return a 1d array.
    def _split_numerical_data(self, X_column, threshold):
        left_index = np.argwhere(X_column <= threshold).flatten()
        right_index = np.argwhere(X_column > threshold).flatten()
        return left_index, right_index
    
    #The below function also splits data but for categorical data
    #Work with data with more than 2 categories
    def _split_categorical_data(self, x_column, category):
        left_index = np.argwhere(x_column == category).flatten()
        right_index = np.argwhere(x_column != category).flatten()
        return left_index, right_index
    
    #Computes the IG for a specific split
    #Return the information gain
    def _information_gain(self, y, X_column, threshold):
        #Calculate parent's entropy
        parent_entropy = self._cross_entropy(y)

        #Create children
        left_index, right_index = self._split_numerical_data(X_column, threshold)

        #If one side is empty, gain is set to 0
        #Explanation: if one child is empty, the split
        # is basically useless
        if len(left_index) == 0 or len(right_index) == 0:
            return 0

        #Calculate left and right entropy
        left_entropy = self._cross_entropy(y[left_index])
        right_entropy = self._cross_entropy(y[right_index])

        #Calculate weighted average
        total_cases = len(y)
        weighted_left = (len(y[left_index]) / total_cases) * left_entropy
        weighted_right = (len(y[right_index]) / total_cases) * right_entropy
        weighted_avg = weighted_left + weighted_right

        #Calculate Information gain
        information_gain = parent_entropy - weighted_avg

        return information_gain
    
    #Return feature's index and threshold that produce the best split for numerical data
    def _best_split_numerical(self, y, X, range_index):
        best_gain = -1
        split_index = None
        split_threshold = None

        #Calculate the average between adjacent cell
        for feature_index in range_index:
            #Take all row from range_idx column
            x_col = X[:, feature_index]

            #threshold = the average of adjacent elements in x_col
            x_col_sorted = np.sort(x_col)
            thresholds_list = (x_col_sorted[:-1] + x_col_sorted[1:]) / 2

            for threshold in thresholds_list:
                info_gain = self._information_gain(y,x_col,threshold)

                #Finding out what is the best feature and the best threshold
                if info_gain > best_gain:
                    best_gain = info_gain
                    split_threshold = threshold
                    split_index = feature_index

        return split_index, split_threshold
    
    #Decide which class should the tree predicts
    def _most_common_label(self, np_labels):
        counter = Counter(np_labels)
        return counter.most_common(1)[0][0]

    #Build the tree recursively
    def _grow_tree(self, y, X, threshold, depth = 0):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))

        #Checking stopping criteria
        if (depth>=self.max_depth or n_labels == 1 or n_samples < self.min_sample_split):
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)

        #Find the best split
        random_range_idx = np.random.choice(n_features, self.n_features, replace=False)
        best_feature, best_threshold = self._best_split_numerical(y,X,random_range_idx)

        #Create child nodes
        left_index, right_index = self._split_numerical_data(X[:,best_feature],best_threshold)
        left = self._grow_tree(y[left_index], X[left_index, :], depth+1)
        right = self._grow_tree(y[right_index], X[right_index, :], depth+1)
        return Node(best_feature, best_threshold, left, right) #return a splitting node

    def fit(self, something):
        pass
    def _traverse_tree(self, something):
        pass
    def predict(self, something):
        pass

    